# 00 - Setup and Configuration
## Sales ETL Pipeline | Medallion Architecture

**Purpose:** Centralised configuration for the Sales ETL Pipeline.  
All paths, table names, database names, and Spark settings are defined here.  
Every downstream notebook runs `%run ./00_setup_and_config` to inherit these.

**Author:** Kathirvel Rajmohan 
**Dataset:** Kaggle Superstore Sales (9,994 rows, 21 columns)  
**Architecture:** Bronze → Silver → Gold → Analytics

## 1. Confirm Spark session is active

In [0]:
print(f"Spark version : {spark.version}")
print(f"Catalog       : {spark.catalog.currentDatabase()}")
print("✅ Spark session is active and ready")

## 2. Define the Unity Catalog / database names

In [0]:
# Database names for each medallion layer
BRONZE_DB   = "sales_bronze"
SILVER_DB   = "sales_silver"
GOLD_DB     = "sales_gold"

# The pipeline name — used in logging and job naming
PIPELINE_NAME = "sales_etl_pipeline"

print(f"Pipeline    : {PIPELINE_NAME}")
print(f"Bronze DB   : {BRONZE_DB}")
print(f"Silver DB   : {SILVER_DB}")
print(f"Gold DB     : {GOLD_DB}")
print("✅ Database config loaded")

## 3. Define source and file path 

In [0]:
SOURCE_PATH = "/Volumes/workspace/default/sales_etl_files/Superstore.csv"  # ← update this

# Delta table storage paths (where Bronze/Silver/Gold tables are written)
BRONZE_PATH = "/delta/sales/bronze"
SILVER_PATH = "/delta/sales/silver"
GOLD_PATH   = "/delta/sales/gold"

# Table names (database.tablename format)
BRONZE_TABLE = f"{BRONZE_DB}.raw_superstore"
SILVER_TABLE = f"{SILVER_DB}.clean_superstore"
FACT_TABLE   = f"{GOLD_DB}.fact_sales"
DIM_CUSTOMER = f"{GOLD_DB}.dim_customer"
DIM_PRODUCT  = f"{GOLD_DB}.dim_product"
DIM_REGION   = f"{GOLD_DB}.dim_region"

print(f"Source file : {SOURCE_PATH}")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Fact table  : {FACT_TABLE}")
print("✅ Path config loaded")


## 4. Create the three medallion databases if they don't already exist

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# Verify they exist
databases = spark.sql("SHOW DATABASES").collect()
db_names = [row[0] for row in databases]

for db in [BRONZE_DB, SILVER_DB, GOLD_DB]:
    status = "✅" if db in db_names else "❌"
    print(f"  {status} {db}")

print("\n✅ All medallion databases confirmed")


## 5. Simple logging utility

In [0]:
from datetime import datetime

def log(level: str, step: str, message: str):
    """
    Lightweight structured logger for pipeline observability.
    
    Args:
        level   : "INFO", "WARN", or "ERROR"
        step    : Which notebook/step this log is from
        message : What happened
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] [{level:5s}] [{step}] {message}")


# Test the logger
log("INFO",  "00_setup", "Pipeline configuration loaded successfully")
log("INFO",  "00_setup", f"Source: {SOURCE_PATH}")
log("WARN",  "00_setup", "Remember to verify SOURCE_PATH matches your upload location")


## 6. Configuration summary - confirm everything is ready

In [0]:
# Cell 8: Configuration summary — confirm everything is ready

print("=" * 55)
print(f"  {PIPELINE_NAME.upper()}")
print("=" * 55)
print(f"  Source file   : {SOURCE_PATH}")
print(f"  Bronze DB     : {BRONZE_DB}  →  {BRONZE_TABLE}")
print(f"  Silver DB     : {SILVER_DB}  →  {SILVER_TABLE}")
print(f"  Gold DB       : {GOLD_DB}")
print(f"    Fact table  : {FACT_TABLE}")
print(f"    Dim tables  : {DIM_CUSTOMER}")
print(f"                  {DIM_PRODUCT}")
print(f"                  {DIM_REGION}")
print("=" * 55)
print("  ✅ Notebook 00 complete — config ready for all layers")
print("=" * 55)